# Детекция ботов по событиям внутри суточного окна

Задача: для каждой `cookie_id` из `data/test.csv` предсказать `score` от 0 до 1 — вероятность того, что кука принадлежит сервису автоматизированного сбора данных. Метрика — Precision при Recall ≥ 70% (`metric.py`).

Структура ноутбука:
1. Настройки и загрузка данных
2. Очистка данных
3. Исследование данных (EDA) и находки
4. Окно наблюдения
5. Признаки
6. Валидация и журнал экспериментов
7. Финальная модель и `submission.csv`

## 1. Настройки и загрузка данных

In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from metric import precision_at_recall  # официальная реализация метрики, ей же считает автопроверка

SEED = 0
VALID_START = '2026-04-17'  # окна начиная с этой даты идут в валидацию (тест лежит позже трейна по времени)

# выдан архив events.csv.gz; поддерживаем и распакованный вариант
EVENTS_PATH = 'data/events.csv.gz' if os.path.exists('data/events.csv.gz') else 'data/events.csv'

In [ ]:
date_cols = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv('data/train.csv', parse_dates=date_cols)
test = pd.read_csv('data/test.csv', parse_dates=date_cols)
events = pd.read_csv(EVENTS_PATH, parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))

## 2. Очистка данных

- `platform` записана в разном регистре (`android` / `Android` / `ANDROID`) и с синонимами (`iphone` = `ios`, `desktop` = `web` — одинаковые User-Agent). Приводим к трём значениям. Исходное написание сохраняем в `platform_raw` только для проверки в EDA.
- Строки файла событий перемешаны, поэтому сортируем по куке и времени: без этого паузы между событиями считаются неверно.

In [ ]:
events['platform_raw'] = events.platform
events['platform'] = events.platform.str.lower().replace({'iphone': 'ios', 'desktop': 'web'})
events = events.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)

events.platform.value_counts()

## 3. Исследование данных (EDA)

### 3.1. Типы событий, платформы, пропуски

In [ ]:
print(events.event_name.value_counts(), '\n')
print(events.groupby('eid').event_name.unique(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

Пропуски объясняются типами событий: `search_query` и `search_page` заполнены только у поиска (~30% событий), `pointer_x/y` отсутствуют у мобильных приложений.

### 3.2. Капча: проверка на утечку

In [ ]:
windows = pd.concat([
    train[['cookie_id', 'window_start_ts', 'window_end_ts']],
    test[['cookie_id', 'window_start_ts', 'window_end_ts']],
])
cap = events[events.event_name == 'captcha_shown'].merge(windows, on='cookie_id')
cap['position'] = np.select(
    [cap.event_ts < cap.window_start_ts, cap.event_ts >= cap.window_end_ts],
    ['до окна', 'после окна'],
    default='внутри окна',
)
cap.position.value_counts()

**Вывод:** все события `captcha_shown` лежат после окна наблюдения. Это информация из будущего (утечка), поэтому признаки по капче не используются. Фильтр по окну их и так отсекает.

### 3.3. Время создания куки: проверка на утечку

In [ ]:
for name, df in [('train', train), ('test', test)]:
    print(name, (df.cookie_created_at >= df.window_end_ts).sum())

**Вывод:** у всех кук `cookie_created_at` раньше конца окна, поэтому возраст куки можно использовать как признак.

## 4. Окно наблюдения

In [ ]:
def events_in_window(events, meta):
    """Оставляет события каждой куки внутри её окна: window_start_ts <= event_ts < window_end_ts."""
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]


ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

### 4.1. Боты и люди: сравнение кандидатов в признаки

In [ ]:
def show_cookie(cookie_id):
    """Показывает события одной куки внутри окна по порядку времени."""
    return ev_tr[ev_tr.cookie_id == cookie_id].sort_values('event_ts')


show_cookie(train[train.target == 1].cookie_id.iloc[0])

In [ ]:
ev = ev_tr.sort_values(['cookie_id', 'event_ts']).copy()
ev['gap'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()

cand = ev.groupby('cookie_id').agg(
    n_events=('event_ts', 'size'),
    first_ts=('event_ts', 'min'),
    last_ts=('event_ts', 'max'),
    gap_median=('gap', 'median'),
    gap_std=('gap', 'std'),
    gap_max=('gap', 'max'),
    max_page=('search_page', 'max'),
    n_queries=('search_query', 'nunique'),
    n_categories=('item_category', 'nunique'),
    n_locations=('item_location', 'nunique'),
    n_ua=('user_agent', 'nunique'),
).reset_index()
cand['duration_min'] = (cand.last_ts - cand.first_ts).dt.total_seconds() / 60
cand = cand.merge(train[['cookie_id', 'target']], on='cookie_id')

cand.drop(columns=['cookie_id', 'first_ts', 'last_ts']).groupby('target').median().T

**Выводы (медианы, 0 — люди, 1 — боты):**
- `gap_median`: 59 с против 25 с — боты кликают быстрее;
- `max_page`: 4 против 7 — боты листают выдачу глубже;
- `n_locations`: 5 против 8 — боты смотрят больше городов;
- `n_events`: 12 против 20 — боты делают больше действий;
- `n_categories`, `n_ua`, `duration_min`, `gap_max` почти не отличаются: боты тоже работают часами и делают длинные перерывы.

In [ ]:
ev_t = ev.merge(train[['cookie_id', 'target']], on='cookie_id')
pd.crosstab(ev_t.target, ev_t.event_name, normalize='index').T.round(3)

**Выводы:** люди чаще добавляют в избранное (6,6% против 2,7%), логинятся (2,1% против 0,8%) и листают фото (12,3% против 7,0%). Боты чаще просматривают объявления и выдачу.

In [ ]:
chk = ev_tr.groupby('cookie_id').platform_raw.nunique().rename('n_platform_spellings').reset_index()
chk = chk.merge(train[['cookie_id', 'target']], on='cookie_id')
chk.groupby('target').n_platform_spellings.agg(
    mean='mean',
    share_multi=lambda s: (s > 1).mean(),
)

**Вывод:** разнобой в написании `platform` есть у 96% людей и 99% ботов, то есть это шум логирования, а не сигнал. Значения нормализованы, признак из написаний не строится.

## 5. Признаки

In [ ]:
def build_features(ev, meta):
    """Собирает по событиям окна одну строку признаков на каждую куку из meta."""
    ev = ev.sort_values(['cookie_id', 'event_ts']).copy()
    ev['gap'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()  # пауза до предыдущего события, сек

    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        # объём и разнообразие
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
        'n_locations': g.item_location.nunique(),
        'n_queries': g.search_query.nunique(),
        # ритм
        'gap_median': g.gap.median(),
        # глубина поиска
        'max_page': g.search_page.max(),
    })

    # доли типов событий у куки
    share = pd.crosstab(ev.cookie_id, ev.event_name, normalize='index')
    share.columns = ['share_' + c for c in share.columns]
    f = f.join(share)

    # признаки из метаданных куки
    f = meta[['cookie_id', 'window_start_ts', 'cookie_created_at']].merge(
        f.reset_index(), on='cookie_id', how='left'
    )
    f['cookie_age_h'] = (f.window_start_ts - f.cookie_created_at).dt.total_seconds() / 3600
    f = f.drop(columns=['window_start_ts', 'cookie_created_at'])
    return f.fillna(0)


X_tr = build_features(ev_tr, train)
X_te = build_features(ev_te, test)
y_tr = train.target.values
feature_cols = [c for c in X_tr.columns if c != 'cookie_id']

print(len(feature_cols), 'признаков')
X_tr.head()

## 6. Валидация

Тест лежит позже трейна по времени, поэтому валидация тоже по времени: обучение на окнах до `VALID_START`, проверка на окнах начиная с этой даты.

In [ ]:
def make_model():
    """Возвращает модель с фиксированными гиперпараметрами и seed."""
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1)


def evaluate(X, y, cols):
    """Обучает модель на ранних окнах и возвращает P@R0.7 на поздних."""
    is_valid = train.window_start_ts.ge(VALID_START).values
    model = make_model()
    model.fit(X.loc[~is_valid, cols], y[~is_valid])
    p_va = model.predict_proba(X.loc[is_valid, cols])[:, 1]
    return precision_at_recall(y[is_valid], p_va)


is_valid = train.window_start_ts.ge(VALID_START).values
print('P@R0.7 на валидации:', round(evaluate(X_tr, y_tr, feature_cols), 4))
print('доля ботов (уровень константы):', round(y_tr[is_valid].mean(), 4))

### Журнал экспериментов

Модель: RandomForest (300 деревьев, `min_samples_leaf=3`, `random_state=0`).

| № | Шаг | Признаки | P@R0.7 |
|---|---|---|---|
| 0 | Константа (всем одинаковый score) | — | 0.0820 |
| 1 | Baseline из quickstart | n_events, item_nunique | 0.1025 |
| 2 | + ритм и поиск | + gap_median, max_page, n_locations, n_queries | 0.2258 |
| 3 | + доли типов событий | + share_* по event_name | 0.3019 |
| 4 | + возраст куки | + cookie_age_h | 0.3500 |

### Идеи на следующие эксперименты

- ритм внутри сессий: доля пауз < 3 с, mean/std/CV пауз < 10 мин, число сессий;
- проверить полные дубликаты событий: удалить и/или сделать долю дублей признаком;
- заполнение `gap_median` для кук с одним событием не нулём;
- разбор `user_agent` на части;
- LightGBM вместо RandomForest;
- несколько временных разрезов валидации.

## 7. Финальная модель и submission.csv

In [ ]:
final_model = make_model()
final_model.fit(X_tr[feature_cols], y_tr)

sub = pd.DataFrame({
    'cookie_id': X_te.cookie_id,
    'score': final_model.predict_proba(X_te[feature_cols])[:, 1],
})

# проверки формата: каждая кука из test ровно один раз, score в [0, 1]
assert len(sub) == len(test)
assert sub.cookie_id.is_unique
assert set(sub.cookie_id) == set(test.cookie_id)
assert sub.score.between(0, 1).all()

sub.to_csv('submission.csv', index=False)
sub.head()